In [ ]:
# ============================================================
# 0) GOOGLE DRIVE BAĞLANTISI
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# # Dataset 3 LFP/Gr SoH Experiments — MAW-GME
#
# Bu notebook, önceki Oxford Dataset 2 kodunun **Dataset 3 LFP/Gr** veri setine uyarlanmış halidir.
#
# Beklenen Google Drive klasörü:
#
# ```text
# /content/drive/MyDrive/Batarya/rpt_data
# ```
#
# Bu klasörde aşağıdaki biçimde RPT CSV dosyaları bulunmalıdır:
#
# ```text
# rpt_cell_51_part0.csv
# rpt_cell_51_part1.csv
# ...
# ```
#
# Bu notebook özellikle örnek makaledeki Dataset 3 için kullanılan **Cell51–Cell55** hücrelerini hedefler. Kod, RPT dosyalarındaki `ref_chg` kısmından ICA/DVA tabanlı sağlık göstergeleri, `ref_dchg` kısmından kapasite ve SoH hesaplar. Ardından önceki deney akışını Dataset 3'e göre çalıştırır:
#
# - Single-HI Polynomial-EKF
# - Approximate paper-style GPR-EKF
# - Multi-HI tabular modeller
# - LSTM/GRU/TCN/Transformer
# - MAW-GME
# - Missing-HI robustness
# - SHAP açıklanabilirlik analizi

In [ ]:
# ============================================================
# PART A - DATASET 3 LFP/Gr RPT DATA FEATURE EXTRACTION
# ============================================================

import os
import re
import sys
import math
import glob
import random
import warnings
import subprocess
import importlib.util
from pathlib import Path

warnings.filterwarnings("ignore")


def pip_install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, imp in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("scikit-learn", "sklearn"),
    ("tqdm", "tqdm"),
    ("openpyxl", "openpyxl"),
]:
    pip_install_if_missing(pkg, imp)

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    def display(obj):
        try:
            print(obj.to_string() if hasattr(obj, "to_string") else obj)
        except Exception:
            print(obj)


# ============================================================
# DRIVE-BASED PROJECT PATHS FOR DATASET 3
# ============================================================
# Görseldeki klasör:
# Drive'im > Batarya > rpt_data
DRIVE_ROOT = "/content/drive/MyDrive/Batarya"

# RPT CSV dosyaları bu klasörde olmalı:
RPT_DATA_DIR = os.path.join(DRIVE_ROOT, "rpt_data")

# Sonuçlar için ayrı proje klasörü:
PROJECT_DIR = os.path.join(DRIVE_ROOT, "Dataset3_LFP_MAW_GME")
FEATURE_DIR = os.path.join(PROJECT_DIR, "features")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")

for _dir in [PROJECT_DIR, FEATURE_DIR, RESULT_DIR]:
    os.makedirs(_dir, exist_ok=True)

FEATURE_OUT_CSV = os.path.join(FEATURE_DIR, "multi_hi_features.csv")

# Örnek makaledeki Dataset 3 için kullanılan hücreler:
TARGET_CELLS = [51, 52, 53, 54, 55]

# LFP için voltaj pencereleri.
# Örnek makalede Dataset 3 için Pmax ölçüm aralığı yaklaşık 3.35-3.50 V olarak verilmişti.
PARTIAL_WINDOWS = {
    "full_27_36": (2.70, 3.60),
    "medium_32_35": (3.20, 3.50),
    "narrow_33_345": (3.30, 3.45),
    "paper_335_350": (3.35, 3.50),
    "very_narrow_335_340": (3.35, 3.40),
}

print("Dataset 3 RPT data directory:", RPT_DATA_DIR)
print("Project directory:", PROJECT_DIR)
print("Feature output CSV:", FEATURE_OUT_CSV)
print("Target cells:", TARGET_CELLS)
print("Partial windows:", PARTIAL_WINDOWS)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def find_col(df, candidates, required=True):
    """
    CSV kolon adlarını küçük farklılıklara karşı güvenli bulur.
    """
    col_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in col_map:
            return col_map[key]

    # Daha gevşek arama
    for c in df.columns:
        c_low = str(c).strip().lower()
        for cand in candidates:
            cand_low = str(cand).strip().lower()
            if cand_low in c_low or c_low in cand_low:
                return c

    if required:
        raise KeyError(f"Column not found. Candidates={candidates}. Existing={list(df.columns)}")
    return None


def to_numeric_series(s):
    return pd.to_numeric(s, errors="coerce")


def parse_cell_and_part(path):
    name = os.path.basename(path)
    m = re.search(r"rpt_cell_(\d+)_part(\d+)\.csv", name, flags=re.IGNORECASE)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def list_rpt_files_for_cell(cell_no, rpt_dir=RPT_DATA_DIR):
    pattern = os.path.join(rpt_dir, f"rpt_cell_{cell_no:02d}_part*.csv")
    files = glob.glob(pattern)

    def sort_key(p):
        _, part = parse_cell_and_part(p)
        return part if part is not None else 999999

    files = sorted(files, key=sort_key)
    return files


def read_cell_rpt_data(cell_no):
    files = list_rpt_files_for_cell(cell_no)
    if len(files) == 0:
        print(f"WARNING: Cell {cell_no} için RPT CSV bulunamadı.")
        return pd.DataFrame()

    parts = []
    for p in files:
        print(f"Reading Cell{cell_no}: {os.path.basename(p)}")
        try:
            part = pd.read_csv(p, low_memory=False)
            part["__source_file"] = os.path.basename(p)
            parts.append(part)
        except Exception as e:
            print("  Could not read:", p, repr(e))

    if len(parts) == 0:
        return pd.DataFrame()

    df = pd.concat(parts, ignore_index=True)

    # Temel kolonları güvenli bul ve standardize et
    col_week = find_col(df, ["Week Number"], required=False)
    col_life = find_col(df, ["Life"], required=False)
    col_date = find_col(df, ["Date (yyyy.mm.dd hh.mm.ss)", "Date"], required=False)
    col_cycle = find_col(df, ["Cycle Number"], required=False)
    col_state = find_col(df, ["State"], required=False)
    col_time = find_col(df, ["Time (s)", "Time"], required=True)
    col_voltage = find_col(df, ["Voltage (V)", "Voltage"], required=True)
    col_current = find_col(df, ["Current (A)", "Current"], required=False)
    col_capacity = find_col(df, ["Capacity (Ah)", "Capacity"], required=True)
    col_step = find_col(df, ["Step Number"], required=False)
    col_segment = find_col(df, ["Segment Key"], required=True)
    col_num_cycles = find_col(df, ["Num Cycles"], required=False)

    rename_map = {
        col_week: "week_number" if col_week else None,
        col_life: "life" if col_life else None,
        col_date: "date" if col_date else None,
        col_cycle: "cycle_number_raw" if col_cycle else None,
        col_state: "state" if col_state else None,
        col_time: "time_s",
        col_voltage: "voltage_v",
        col_current: "current_a" if col_current else None,
        col_capacity: "capacity_ah",
        col_step: "step_number" if col_step else None,
        col_segment: "segment_key",
        col_num_cycles: "num_cycles" if col_num_cycles else None,
    }
    rename_map = {k: v for k, v in rename_map.items() if k is not None and v is not None}
    df = df.rename(columns=rename_map)

    for c in ["week_number", "cycle_number_raw", "time_s", "voltage_v", "current_a", "capacity_ah", "step_number", "num_cycles"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if "segment_key" in df.columns:
        df["segment_key"] = df["segment_key"].astype(str).str.strip().str.lower()

    if "state" in df.columns:
        df["state"] = df["state"].astype(str).str.strip()

    if "date" in df.columns:
        df["date_parsed"] = pd.to_datetime(df["date"], format="%Y.%m.%d %H.%M.%S", errors="coerce")
    else:
        df["date_parsed"] = pd.NaT

    return df


def segment_filter(df, segment_name):
    """
    RPT içinden ref_chg veya ref_dchg bölgesini seçer.
    Segment Key yoksa Current/State üzerinden fallback dener.
    """
    segment_name = segment_name.lower()
    if "segment_key" in df.columns:
        out = df[df["segment_key"].astype(str).str.lower().eq(segment_name)].copy()
        if len(out) > 0:
            return out

    # Fallback
    if segment_name == "ref_chg":
        if "current_a" in df.columns:
            out = df[df["current_a"] > 0].copy()
            if len(out) > 0:
                return out
        if "state" in df.columns:
            return df[df["state"].astype(str).str.lower().str.contains("chg", na=False) &
                      ~df["state"].astype(str).str.lower().str.contains("dchg", na=False)].copy()

    if segment_name == "ref_dchg":
        if "current_a" in df.columns:
            out = df[df["current_a"] < 0].copy()
            if len(out) > 0:
                return out
        if "state" in df.columns:
            return df[df["state"].astype(str).str.lower().str.contains("dchg", na=False)].copy()

    return pd.DataFrame(columns=df.columns)


def safe_capacity_from_ref_dchg(ref_dchg):
    if ref_dchg is None or len(ref_dchg) < 10:
        return np.nan
    q = to_numeric_series(ref_dchg["capacity_ah"]).dropna().values.astype(float)
    if len(q) < 10:
        return np.nan
    cap = float(abs(np.nanmax(q) - np.nanmin(q)))
    # LFP nominal 1.2 Ah. RPT capacity biraz bunun çevresinde olmalı.
    if not np.isfinite(cap) or cap <= 0:
        return np.nan
    return cap


def get_charge_arrays_from_ref_chg(ref_chg):
    if ref_chg is None or len(ref_chg) < 30:
        return np.array([]), np.array([]), np.array([]), np.array([])

    v = to_numeric_series(ref_chg["voltage_v"]).values.astype(float)
    q = to_numeric_series(ref_chg["capacity_ah"]).values.astype(float)
    t = to_numeric_series(ref_chg["time_s"]).values.astype(float)

    # Bu dataset README'ye göre sıcaklık kolonu içermiyor. Eksik bırakıyoruz.
    T = np.full(len(v), np.nan)

    n = min(len(v), len(q), len(t))
    v, q, t, T = v[:n], q[:n], t[:n], T[:n]

    mask = np.isfinite(v) & np.isfinite(q)
    v, q, t, T = v[mask], q[mask], t[mask], T[mask]

    # Zaman sırası ile sırala; sonra voltaja göre gruplayacağız.
    if len(t) == len(v) and np.isfinite(t).sum() > 0:
        order = np.argsort(t)
        v, q, t, T = v[order], q[order], t[order], T[order]

    return v, q, t, T


def compute_ica_and_dva_window_from_ref_chg(ref_chg, v_window, n_grid=500, smooth_window=31, smooth_poly=3):
    v, q, t, T = get_charge_arrays_from_ref_chg(ref_chg)
    if len(v) < 30:
        return None, None, None, None, None

    lo, hi = v_window
    mask = np.isfinite(v) & np.isfinite(q) & (v >= lo) & (v <= hi)
    if mask.sum() < 30:
        return None, None, None, None, None

    v_w, q_w = v[mask], q[mask]
    t_w = t[mask] if len(t) == len(v) else np.full(mask.sum(), np.nan)
    T_w = T[mask] if len(T) == len(v) else np.full(mask.sum(), np.nan)

    df_tmp = pd.DataFrame({"v": v_w, "q": q_w, "t": t_w, "T": T_w}).replace([np.inf, -np.inf], np.nan).dropna(subset=["v", "q"])
    if len(df_tmp) < 30:
        return None, None, None, None, None

    # ICA için voltaj gridine geçiyoruz. CV fazındaki aynı voltajlı çoklu noktaları gruplayarak gürültüyü azaltıyoruz.
    df_tmp["v_round"] = df_tmp["v"].round(4)
    df_g = df_tmp.groupby("v_round", as_index=False).agg(q=("q", "mean"), t=("t", "mean"), T=("T", "mean"))
    df_g = df_g.rename(columns={"v_round": "v"}).sort_values("v")

    v_unique = df_g["v"].values.astype(float)
    q_unique = df_g["q"].values.astype(float)

    if len(v_unique) < 20:
        return None, None, None, None, None

    v_grid = np.linspace(v_unique.min(), v_unique.max(), n_grid)
    q_grid = np.interp(v_grid, v_unique, q_unique)

    sw = smooth_window
    if sw >= len(q_grid):
        sw = len(q_grid) - 1
    if sw % 2 == 0:
        sw -= 1
    if sw < 7:
        sw = 7

    try:
        q_smooth = savgol_filter(q_grid, sw, smooth_poly)
    except Exception:
        q_smooth = q_grid.copy()

    # dQ/dV and dV/dQ
    ic = np.gradient(q_smooth, v_grid)
    try:
        ic = savgol_filter(ic, sw, smooth_poly)
    except Exception:
        pass

    # DVA = dV/dQ. q_grid monoton olmayabilir; küçük bölümlerde inf oluşabilir.
    with np.errstate(divide="ignore", invalid="ignore"):
        dva = np.gradient(v_grid, q_smooth)
    try:
        dva = savgol_filter(dva, sw, smooth_poly)
    except Exception:
        pass

    ic[~np.isfinite(ic)] = np.nan
    dva[~np.isfinite(dva)] = np.nan

    raw_df = df_tmp.copy()
    return v_grid, q_grid, ic, dva, raw_df


def peak_width_half_max(v_grid, ic):
    mask = np.isfinite(v_grid) & np.isfinite(ic) & (ic > 0)
    if mask.sum() < 5:
        return np.nan
    v = v_grid[mask]
    y = ic[mask]
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    half = 0.5 * ymax
    above = y >= half
    if above.sum() < 2:
        return np.nan
    return float(v[above].max() - v[above].min())


def extract_multi_hi_for_window_from_ref_chg(ref_chg, v_window):
    v_grid, q_grid, ic, dva, raw_df = compute_ica_and_dva_window_from_ref_chg(ref_chg, v_window)

    out = {
        "pmax": np.nan,
        "pmax_voltage": np.nan,
        "ic_area": np.nan,
        "ic_mean": np.nan,
        "ic_std": np.nan,
        "peak_width": np.nan,
        "q_delta_window": np.nan,
        "charge_time_window": np.nan,
        "dva_mean": np.nan,
        "dva_std": np.nan,
        "dva_abs_mean": np.nan,
        "temp_mean_window": np.nan,
        "temp_max_window": np.nan,
    }

    if v_grid is None:
        return out

    mask_ic = np.isfinite(ic) & (ic > 0)
    if mask_ic.sum() >= 5:
        ic_pos = ic[mask_ic]
        v_pos = v_grid[mask_ic]

        # Gürültüden gelen aşırı tekil değerleri bastır
        upper = np.nanpercentile(ic_pos, 99.5)
        ic_clip = np.clip(ic_pos, None, upper)

        idx = int(np.nanargmax(ic_clip))
        out["pmax"] = float(ic_clip[idx])
        out["pmax_voltage"] = float(v_pos[idx])
        out["ic_area"] = float(np.trapz(ic_clip, v_pos))
        out["ic_mean"] = float(np.nanmean(ic_clip))
        out["ic_std"] = float(np.nanstd(ic_clip))
        out["peak_width"] = peak_width_half_max(v_grid, ic)

    if q_grid is not None and np.isfinite(q_grid).sum() > 5:
        out["q_delta_window"] = float(np.nanmax(q_grid) - np.nanmin(q_grid))

    if raw_df is not None and len(raw_df) > 0:
        if "t" in raw_df.columns and raw_df["t"].notna().sum() > 2:
            out["charge_time_window"] = float(raw_df["t"].max() - raw_df["t"].min())
        if "T" in raw_df.columns and raw_df["T"].notna().sum() > 2:
            out["temp_mean_window"] = float(raw_df["T"].mean())
            out["temp_max_window"] = float(raw_df["T"].max())

    if dva is not None:
        mask_dva = np.isfinite(dva)
        if mask_dva.sum() > 5:
            dva_sel = dva[mask_dva]
            lo, hi = np.nanpercentile(dva_sel, [1, 99])
            dva_clip = np.clip(dva_sel, lo, hi)
            out["dva_mean"] = float(np.nanmean(dva_clip))
            out["dva_std"] = float(np.nanstd(dva_clip))
            out["dva_abs_mean"] = float(np.nanmean(np.abs(dva_clip)))

    return out


def detect_soh_outliers(df, threshold=0.04):
    df = df.sort_values(["cell", "cycle_number"]).copy()
    df["soh_rolling_median"] = np.nan
    df["soh_outlier"] = False
    df["soh_delta"] = np.nan

    for cell_name, g in df.groupby("cell"):
        idx = g.index
        s = g["soh"].astype(float)
        med = s.rolling(window=5, center=True, min_periods=2).median()
        delta = s.diff()
        df.loc[idx, "soh_rolling_median"] = med.values
        df.loc[idx, "soh_delta"] = delta.values

        # Ani ve fiziksel olarak şüpheli düşüş/zıplama noktaları
        out = (s - med).abs() > threshold
        df.loc[idx, "soh_outlier"] = out.values

    return df


def build_dataset3_multi_hi_features():
    if not os.path.isdir(RPT_DATA_DIR):
        raise FileNotFoundError(
            f"RPT_DATA_DIR bulunamadı: {RPT_DATA_DIR}\n"
            "Google Drive içinde Batarya/rpt_data klasörünü kontrol edin."
        )

    rows = []

    for cell_no in TARGET_CELLS:
        cell_name = f"Cell{cell_no}"
        df_cell = read_cell_rpt_data(cell_no)

        if df_cell.empty:
            print(f"Skipping {cell_name}: no data.")
            continue

        # RPT grupları. Week Number varsa onu kullanıyoruz.
        if "week_number" in df_cell.columns and df_cell["week_number"].notna().sum() > 0:
            group_col = "week_number"
        else:
            # Fallback: step/date bazlı değilse tüm dosyayı tek grup yapar; bu istenmez ama hata vermesin.
            group_col = "__rpt_group"
            df_cell[group_col] = 0

        # Sıralama için tarih veya week number
        if "date_parsed" in df_cell.columns and df_cell["date_parsed"].notna().sum() > 0:
            order_df = df_cell.groupby(group_col)["date_parsed"].min().reset_index().sort_values("date_parsed")
            group_keys = order_df[group_col].tolist()
        else:
            group_keys = sorted(df_cell[group_col].dropna().unique().tolist())

        cumulative_cycles = 0.0
        rpt_counter = 0

        for key in tqdm(group_keys, desc=f"Feature extraction {cell_name}"):
            g = df_cell[df_cell[group_col].eq(key)].copy()
            if len(g) < 50:
                continue

            ref_chg = segment_filter(g, "ref_chg")
            ref_dchg = segment_filter(g, "ref_dchg")

            cap_ah = safe_capacity_from_ref_dchg(ref_dchg)
            if not np.isfinite(cap_ah):
                continue

            # Num Cycles non-cumulative olduğu için kümülatif hale getiriyoruz.
            cycle_number = cumulative_cycles
            inc = np.nan
            if "num_cycles" in g.columns:
                vals = pd.to_numeric(g["num_cycles"], errors="coerce").dropna()
                if len(vals) > 0:
                    inc = float(vals.max())

            row = {
                "cell": cell_name,
                "cycle_name": f"rpt_{rpt_counter:04d}",
                "rpt_group": key,
                "rpt_index": rpt_counter,
                "cycle_number": cycle_number,
                "num_cycles_increment": inc,
                "capacity_mAh": cap_ah * 1000.0,
                "capacity_Ah": cap_ah,
                # Bu dataset README'ye göre sıcaklık kolonu içermiyor.
                "temp_mean": np.nan,
                "temp_max": np.nan,
            }

            for tag, win in PARTIAL_WINDOWS.items():
                feats = extract_multi_hi_for_window_from_ref_chg(ref_chg, win)
                for k, v in feats.items():
                    row[f"{k}_{tag}"] = v

            rows.append(row)
            rpt_counter += 1

            if np.isfinite(inc) and inc >= 0:
                cumulative_cycles += inc
            else:
                # Num Cycles yoksa yaklaşık her RPT ~100 cycle varsayımı
                cumulative_cycles += 100.0

    features = pd.DataFrame(rows)

    if features.empty:
        raise RuntimeError(
            "Hiç feature üretilemedi. RPT dosyalarını, Segment Key değerlerini ve hedef hücreleri kontrol edin."
        )

    features = features.sort_values(["cell", "cycle_number"]).reset_index(drop=True)

    # SoH ve normalized Pmax
    features["soh"] = np.nan

    for cell_name, g in features.groupby("cell"):
        idx = g.index
        caps = g["capacity_mAh"].dropna()
        if len(caps) == 0:
            continue
        first_capacity = float(caps.iloc[0])
        features.loc[idx, "soh"] = features.loc[idx, "capacity_mAh"] / first_capacity

        for tag in PARTIAL_WINDOWS.keys():
            pcol = f"pmax_{tag}"
            ncol = f"pmax_norm_{tag}"
            if pcol in features.columns and g[pcol].dropna().shape[0] > 0:
                first_pmax = float(g[pcol].dropna().iloc[0])
                if np.isfinite(first_pmax) and abs(first_pmax) > 1e-12:
                    features.loc[idx, ncol] = features.loc[idx, pcol] / first_pmax
                else:
                    features.loc[idx, ncol] = np.nan
            else:
                features.loc[idx, ncol] = np.nan

    features = detect_soh_outliers(features, threshold=0.04)

    return features

In [ ]:
print("\n================ PART A: DATASET 3 LFP DATA + FEATURE EXTRACTION ================")
multi_features = build_dataset3_multi_hi_features()
print("Multi-HI feature table shape:", multi_features.shape)
display(multi_features.head())

multi_features.to_csv(FEATURE_OUT_CSV, index=False)
multi_features.to_csv("/content/multi_hi_features.csv", index=False)

print("Saved feature file to Google Drive:", FEATURE_OUT_CSV)
print("Temporary Colab copy:", "/content/multi_hi_features.csv")

# Quick correlation file for convenience
corr_rows = []
for col in multi_features.select_dtypes(include=[np.number]).columns:
    if col == "soh":
        continue
    valid = multi_features[["soh", col]].dropna()
    if len(valid) >= 10:
        corr = valid["soh"].corr(valid[col])
        corr_rows.append({"feature": col, "corr_with_soh": corr, "abs_corr": abs(corr), "valid_count": len(valid)})

corr_df = pd.DataFrame(corr_rows).sort_values("abs_corr", ascending=False)
corr_path = os.path.join(FEATURE_DIR, "multi_hi_correlation_with_soh.csv")
corr_df.to_csv(corr_path, index=False)
print("Saved correlation file to Google Drive:", corr_path)
print("Top correlated features with SoH:")
print(corr_df.head(20).to_string(index=False))

print("\nCell-level summary:")
summary = multi_features.groupby("cell").agg(
    n_rpt=("cycle_number", "count"),
    first_cycle=("cycle_number", "min"),
    last_cycle=("cycle_number", "max"),
    first_capacity_mAh=("capacity_mAh", "first"),
    last_capacity_mAh=("capacity_mAh", "last"),
    first_soh=("soh", "first"),
    last_soh=("soh", "last"),
    outlier_count=("soh_outlier", "sum"),
).reset_index()
display(summary)

In [ ]:
# ============================================================
# PART B - ADVANCED EXPERIMENTS
# ============================================================



# ============================================================
# ADVANCED DATASET 3 LFP BATTERY SOH EXPERIMENTS
# Multi-HI + Partial Window + Leakage Cleaning + Single-HI EKF
# Approx. Paper-style GPR-EKF + ML models + LSTM/GRU/TCN/Transformer
# Missing-HI robustness + SHAP plots
# ============================================================

# ------------------------------------------------------------
# 0) COLAB INSTALLS
# ------------------------------------------------------------
import sys, subprocess, importlib.util, os, warnings, math, json, random

def pip_install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core packages
for pkg, imp in [
    ("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "sklearn"),
    ("matplotlib", "matplotlib"), ("seaborn", "seaborn"),
    ("xgboost", "xgboost"), ("lightgbm", "lightgbm"), ("catboost", "catboost"),
    ("shap", "shap"), ("openpyxl", "openpyxl")
]:
    try:
        pip_install_if_missing(pkg, imp)
    except Exception as e:
        print(f"WARNING: {pkg} could not be installed/imported: {e}")

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(obj):
        try:
            print(obj.to_string() if hasattr(obj, "to_string") else obj)
        except Exception:
            print(obj)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    from catboost import CatBoostRegressor
    CAT_AVAILABLE = True
except Exception:
    CAT_AVAILABLE = False

In [ ]:
# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# If deep models are slow, set this to False.
RUN_DEEP_MODELS = True
DEEP_EPOCHS = 80
DEEP_SEQ_LEN = 5
DEEP_BATCH_SIZE = 32
DEEP_LR = 1e-3
DEEP_PATIENCE = 15

# Missing-HI robustness settings
MISSING_RANDOM_SEEDS = list(range(10))
MISSING_PROBS = [0.0, 0.2, 0.4, 0.6, 0.8]

# If True, outlier rows will be excluded in one of the repeated analyses.
EXCLUDE_OUTLIER_OPTIONS = [False, True]

# Partial-charging windows already encoded in multi_hi_features.csv
WINDOWS = {
    "full_27_36": "Full 2.70-3.60 V",
    "medium_32_35": "Medium 3.20-3.50 V",
    "narrow_33_345": "Narrow 3.30-3.45 V",
    "paper_335_350": "Paper/Pmax 3.35-3.50 V",
    "very_narrow_335_340": "Very narrow 3.35-3.40 V",
}

# Kalıcı sonuç klasörü: Google Drive
# Bu hücre Part A çalıştırılmadan tek başına çalıştırılırsa PROJECT_DIR'i tekrar tanımlar.
DRIVE_ROOT = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/Batarya")
PROJECT_DIR = globals().get("PROJECT_DIR", os.path.join(DRIVE_ROOT, "Dataset3_LFP_MAW_GME"))
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)
print("Result directory:", RESULT_DIR)

In [ ]:
# ------------------------------------------------------------
# 2) LOAD MULTI-HI FEATURE FILE
# ------------------------------------------------------------
# This code continues from the previous notebook.
# It expects multi_hi_features.csv produced by your previous feature extraction.

DRIVE_ROOT = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/Batarya")
PROJECT_DIR = globals().get("PROJECT_DIR", os.path.join(DRIVE_ROOT, "Dataset3_LFP_MAW_GME"))
FEATURE_DIR = globals().get("FEATURE_DIR", os.path.join(PROJECT_DIR, "features"))
FEATURE_OUT_CSV = globals().get("FEATURE_OUT_CSV", os.path.join(FEATURE_DIR, "multi_hi_features.csv"))

CANDIDATE_FEATURE_PATHS = [
    FEATURE_OUT_CSV,  # preferred: Google Drive project feature file
    os.path.join(FEATURE_DIR, "multi_hi_features.csv"),
    "/content/multi_hi_features.csv",  # temporary Colab copy
    "multi_hi_features.csv",
    "/content/dataset3_lfp/multi_hi_features.csv",
    "/mnt/data/multi_hi_features.csv",   # local ChatGPT environment fallback
]

FEATURE_CSV = None
for p in CANDIDATE_FEATURE_PATHS:
    if os.path.exists(p):
        FEATURE_CSV = p
        break

# If you have a DataFrame variable named multi_hi_features in the notebook,
# you can uncomment these lines instead:
# df = multi_hi_features.copy()

if FEATURE_CSV is None:
    raise FileNotFoundError(
        "multi_hi_features.csv bulunamadı. Önce önceki Multi-HI feature extraction kodunu çalıştırın "
        "veya FEATURE_CSV değişkenini dosya yoluna göre düzenleyin."
    )

print("Loading:", FEATURE_CSV)
df = pd.read_csv(FEATURE_CSV)
print("Feature table shape:", df.shape)
print("Cells:", sorted(df["cell"].unique()))

# Ensure sorting
df = df.sort_values(["cell", "cycle_number"]).reset_index(drop=True)

In [ ]:
# ------------------------------------------------------------
# 3) METRICS AND UTILS
# ------------------------------------------------------------
def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def safe_r2(y_true, y_pred):
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return np.nan

def regression_metrics(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": safe_r2(y_true, y_pred),
    }

# ------------------------------------------------------------
# 4) FEATURE LEAKAGE CLEANING
# ------------------------------------------------------------
# These columns must NOT be used as model inputs because they are target-derived
# or directly define SoH.
LEAKAGE_COLUMNS_EXACT = {
    "soh", "capacity_mAh", "soh_rolling_median", "soh_delta", "soh_outlier",
    "cycle_name", "cell"
}

# We keep cycle_number because it is equivalent to EFC/cycle age and is available in practice.
# If you want to test without cycle information, set USE_CYCLE_NUMBER=False.
USE_CYCLE_NUMBER = True
USE_GLOBAL_TEMPERATURE = True

# These are acceptable because they come from charge curves / voltage windows.
# q_delta_window is a partial capacity-like feature within a voltage window, not full capacity.

def get_window_feature_cols(dataframe, window, use_cycle_number=True, use_global_temperature=True):
    """
    Return leakage-clean feature columns for a given partial-charging window.
    Only uses features from the selected window + optional cycle/temp.
    """
    suffix = "_" + window
    cols = []

    for c in dataframe.columns:
        if c in LEAKAGE_COLUMNS_EXACT:
            continue
        if c == "cycle_number" and use_cycle_number:
            cols.append(c)
        elif c in ["temp_mean", "temp_max"] and use_global_temperature:
            cols.append(c)
        elif c.endswith(suffix):
            cols.append(c)
        elif c == f"pmax_norm_{window}":
            cols.append(c)

    # Remove duplicates while preserving order
    cols = list(dict.fromkeys(cols))

    # Final leakage safety check
    bad = [c for c in cols if c in LEAKAGE_COLUMNS_EXACT or c.startswith("soh") or c == "capacity_mAh"]
    if bad:
        raise ValueError(f"Leakage columns found in feature set: {bad}")
    return cols

# Show feature columns per window
for w in WINDOWS:
    print("\n", w, "features:")
    print(get_window_feature_cols(df, w))

# Save leakage-clean feature documentation
feature_doc_rows = []
for w in WINDOWS:
    for c in get_window_feature_cols(df, w):
        feature_doc_rows.append({"window": w, "feature": c})
pd.DataFrame(feature_doc_rows).to_csv(os.path.join(RESULT_DIR, "leakage_clean_feature_sets.csv"), index=False)

In [ ]:
# ------------------------------------------------------------
# 5) TABULAR MODEL DEFINITIONS
# ------------------------------------------------------------
def make_tabular_models():
    models = {}

    # Linear models
    models["Ridge"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ])
    models["ElasticNet"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.001, l1_ratio=0.2, random_state=RANDOM_STATE, max_iter=10000)),
    ])
    models["SVR-RBF"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=10.0, gamma="scale", epsilon=0.001)),
    ])
    models["KNN"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5, weights="distance")),
    ])

    # Tree and boosting models
    models["RandomForest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
        )),
    ])
    models["ExtraTrees"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=500, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1
        )),
    ])
    models["HistGB"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.03, l2_regularization=0.01, random_state=RANDOM_STATE
        )),
    ])

    if XGB_AVAILABLE:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=600, max_depth=3, learning_rate=0.02,
                subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                random_state=RANDOM_STATE, objective="reg:squarederror", n_jobs=-1
            )),
        ])

    if LGBM_AVAILABLE:
        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMRegressor(
                n_estimators=600, max_depth=-1, learning_rate=0.02,
                num_leaves=15, subsample=0.9, colsample_bytree=0.9,
                reg_lambda=1.0, random_state=RANDOM_STATE, verbose=-1
            )),
        ])

    if CAT_AVAILABLE:
        models["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=600, depth=4, learning_rate=0.02,
                loss_function="RMSE", random_seed=RANDOM_STATE, verbose=False
            )),
        ])

    return models

# ------------------------------------------------------------
# 6) LEAVE-ONE-CELL-OUT TABULAR MULTI-HI EVALUATION
# ------------------------------------------------------------
def evaluate_tabular_multi_hi(dataframe, windows=WINDOWS, exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS):
    rows = []
    models = make_tabular_models()
    all_cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for window in windows:
            feature_cols = get_window_feature_cols(data_eval, window, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
            required = ["cell", "soh"] + feature_cols
            data_w = data_eval.dropna(subset=["cell", "soh"]).copy()

            for test_cell in all_cells:
                train_df = data_w[data_w["cell"] != test_cell].copy()
                test_df = data_w[data_w["cell"] == test_cell].copy()

                if len(test_df) < 5 or len(train_df) < 20:
                    continue

                X_train = train_df[feature_cols]
                y_train = train_df["soh"].values.astype(float)
                X_test = test_df[feature_cols]
                y_test = test_df["soh"].values.astype(float)

                for model_name, model in models.items():
                    try:
                        m = clone(model)
                        m.fit(X_train, y_train)
                        pred = m.predict(X_test)
                        pred = np.clip(pred, 0.5, 1.05)
                        met = regression_metrics(y_test, pred)
                        rows.append({
                            "experiment": "MultiHI-Tabular",
                            "window": window,
                            "test_cell": test_cell,
                            "model": model_name,
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(train_df),
                            "n_test": len(test_df),
                            "n_features": len(feature_cols),
                            **met,
                        })
                    except Exception as e:
                        print(f"FAILED tabular {model_name}, {window}, {test_cell}: {e}")
    return pd.DataFrame(rows)

multi_hi_tabular_results = evaluate_tabular_multi_hi(df)
multi_hi_tabular_results.to_csv(os.path.join(RESULT_DIR, "multi_hi_tabular_leave_one_cell_results.csv"), index=False)
print("\nMulti-HI tabular results:")
display(multi_hi_tabular_results.head())

# Summary tables
multi_hi_tabular_summary = (
    multi_hi_tabular_results
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
multi_hi_tabular_summary.to_csv(os.path.join(RESULT_DIR, "multi_hi_tabular_summary_mean_std.csv"), index=False)
display(multi_hi_tabular_summary)

In [ ]:
# ------------------------------------------------------------
# 7) SINGLE-HI POLYNOMIAL EKF BASELINE
# ------------------------------------------------------------
def poly_derivative_value(coef, x):
    return np.polyval(np.polyder(coef), x)

def fit_single_hi_poly_models(train_df, window):
    """
    Prediction model: SoH = f(cycle_number), degree 2
    Measurement model: pmax_norm_window = h(SoH), degree 3
    """
    z_col = f"pmax_norm_{window}"
    tr = train_df.dropna(subset=["cycle_number", "soh", z_col]).copy()
    u = tr["cycle_number"].values.astype(float)
    x = tr["soh"].values.astype(float)
    z = tr[z_col].values.astype(float)

    pred_coef = np.polyfit(u, x, deg=2)
    meas_coef = np.polyfit(x, z, deg=3)

    pred_res = x - np.polyval(pred_coef, u)
    meas_res = z - np.polyval(meas_coef, x)
    q = max(np.var(pred_res), 1e-7)
    r = max(np.var(meas_res), 1e-7)
    return pred_coef, meas_coef, q, r

def single_hi_poly_ekf_predict(test_df, window, pred_coef, meas_coef, q, r, P0=0.01):
    z_col = f"pmax_norm_{window}"
    te = test_df.sort_values("cycle_number").dropna(subset=["cycle_number", "soh", z_col]).copy()
    preds = []
    P = P0
    x = 1.0  # BOL normalized initial health; common assumption for fresh cell

    for _, row in te.iterrows():
        u = float(row["cycle_number"])
        z = float(row[z_col])

        # Prediction from cycle degradation trend
        x_pred = float(np.polyval(pred_coef, u))
        P_pred = P + q

        # Measurement update using normalized Pmax
        z_pred = float(np.polyval(meas_coef, x_pred))
        H = float(poly_derivative_value(meas_coef, x_pred))
        S = H * P_pred * H + r
        if S <= 1e-12 or abs(H) < 1e-12:
            x = x_pred
            P = P_pred
        else:
            K = P_pred * H / S
            x = x_pred + K * (z - z_pred)
            P = (1.0 - K * H) * P_pred
        x = float(np.clip(x, 0.5, 1.05))
        preds.append(x)
    return te, np.array(preds)

def evaluate_single_hi_poly_ekf(dataframe, windows=WINDOWS, exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS):
    rows = []
    all_cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for window in windows:
            z_col = f"pmax_norm_{window}"
            if z_col not in data_eval.columns:
                continue
            for test_cell in all_cells:
                train_df = data_eval[data_eval["cell"] != test_cell].copy()
                test_df = data_eval[data_eval["cell"] == test_cell].copy()
                try:
                    pred_coef, meas_coef, q, r = fit_single_hi_poly_models(train_df, window)
                    te, pred = single_hi_poly_ekf_predict(test_df, window, pred_coef, meas_coef, q, r)
                    y_true = te["soh"].values.astype(float)
                    met = regression_metrics(y_true, pred)
                    rows.append({
                        "experiment": "SingleHI-Polynomial-EKF",
                        "window": window,
                        "test_cell": test_cell,
                        "model": "SingleHI-Polynomial-EKF",
                        "exclude_outliers": exclude_outliers,
                        "n_train": len(train_df),
                        "n_test": len(te),
                        "n_features": 1,
                        **met,
                    })
                except Exception as e:
                    print(f"FAILED SingleHI EKF {window}, {test_cell}: {e}")
    return pd.DataFrame(rows)

single_hi_ekf_results_v2 = evaluate_single_hi_poly_ekf(df)
single_hi_ekf_results_v2.to_csv(os.path.join(RESULT_DIR, "single_hi_polynomial_ekf_leave_one_cell_results.csv"), index=False)
display(single_hi_ekf_results_v2.head())

single_hi_ekf_summary_v2 = (
    single_hi_ekf_results_v2
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
single_hi_ekf_summary_v2.to_csv(os.path.join(RESULT_DIR, "single_hi_polynomial_ekf_summary_mean_std.csv"), index=False)
display(single_hi_ekf_summary_v2)

In [ ]:
# ------------------------------------------------------------
# 8) APPROXIMATE PAPER-STYLE GPR-EKF BASELINE
# ------------------------------------------------------------
def make_gpr_pipeline():
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e4)) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e1))
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", GaussianProcessRegressor(
            kernel=kernel, alpha=1e-7, normalize_y=True,
            n_restarts_optimizer=1, random_state=RANDOM_STATE
        )),
    ])

def gpr_predict_mean_std(pipe, X):
    X = np.asarray(X, dtype=float)
    Xt = pipe.named_steps["imputer"].transform(X)
    Xt = pipe.named_steps["scaler"].transform(Xt)
    return pipe.named_steps["model"].predict(Xt, return_std=True)

def build_transition_training_data(train_df):
    Xs, ys = [], []
    for cell, g in train_df.sort_values(["cell", "cycle_number"]).groupby("cell"):
        g = g.dropna(subset=["soh", "cycle_number"]).copy()
        arr_soh = g["soh"].values.astype(float)
        arr_u = g["cycle_number"].values.astype(float)
        for i in range(len(g) - 1):
            x_k = arr_soh[i]
            u_k = arr_u[i]
            du = arr_u[i+1] - arr_u[i]
            y_next = arr_soh[i+1]
            Xs.append([x_k, u_k, du])
            ys.append(y_next)
    return np.asarray(Xs, dtype=float), np.asarray(ys, dtype=float)

def numerical_derivative_measurement_gpr(meas_gpr, x, delta=1e-4):
    m1, _ = gpr_predict_mean_std(meas_gpr, [[x + delta]])
    m2, _ = gpr_predict_mean_std(meas_gpr, [[x - delta]])
    return float((m1[0] - m2[0]) / (2 * delta))

def fit_approx_paper_gpr_ekf(train_df, window):
    z_col = f"pmax_norm_{window}"
    tr = train_df.dropna(subset=["soh", "cycle_number", z_col]).copy()

    # Prediction GPR: [SoH_k, u_k, delta_u] -> SoH_{k+1}
    X_pred, y_pred = build_transition_training_data(tr)
    pred_gpr = make_gpr_pipeline()
    pred_gpr.fit(X_pred, y_pred)

    # Measurement GPR: SoH_k -> normalized Pmax_k
    X_meas = tr[["soh"]].values.astype(float)
    y_meas = tr[z_col].values.astype(float)
    meas_gpr = make_gpr_pipeline()
    meas_gpr.fit(X_meas, y_meas)
    return pred_gpr, meas_gpr

def approx_paper_gpr_ekf_predict(test_df, window, pred_gpr, meas_gpr, P0=0.01):
    z_col = f"pmax_norm_{window}"
    te = test_df.sort_values("cycle_number").dropna(subset=["cycle_number", "soh", z_col]).copy()
    u_values = te["cycle_number"].values.astype(float)
    z_values = te[z_col].values.astype(float)

    preds = []
    P = P0
    x = 1.0  # Beginning-of-life normalized SoH assumption
    u_prev = float(u_values[0])

    for i, (u, z) in enumerate(zip(u_values, z_values)):
        if i == 0:
            du = 0.0
        else:
            du = float(u - u_prev)

        # Prediction step: [updated previous x, previous u, delta_u] -> predicted current x
        mean_pred, std_pred = gpr_predict_mean_std(pred_gpr, [[x, u_prev, du]])
        x_pred = float(mean_pred[0])
        q = max(float(std_pred[0]) ** 2, 1e-7)
        P_pred = P + q

        # Measurement update: h(x) -> normalized Pmax
        mean_z, std_z = gpr_predict_mean_std(meas_gpr, [[x_pred]])
        z_pred = float(mean_z[0])
        r = max(float(std_z[0]) ** 2, 1e-7)
        H = numerical_derivative_measurement_gpr(meas_gpr, x_pred)

        S = H * P_pred * H + r
        if S <= 1e-12 or abs(H) < 1e-12:
            x = x_pred
            P = P_pred
        else:
            K = P_pred * H / S
            x = x_pred + K * (z - z_pred)
            P = (1.0 - K * H) * P_pred

        x = float(np.clip(x, 0.5, 1.05))
        preds.append(x)
        u_prev = float(u)

    return te, np.array(preds)

def evaluate_approx_paper_gpr_ekf(dataframe, windows=WINDOWS, exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS):
    rows = []
    all_cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for window in windows:
            for test_cell in all_cells:
                train_df = data_eval[data_eval["cell"] != test_cell].copy()
                test_df = data_eval[data_eval["cell"] == test_cell].copy()
                try:
                    pred_gpr, meas_gpr = fit_approx_paper_gpr_ekf(train_df, window)
                    te, pred = approx_paper_gpr_ekf_predict(test_df, window, pred_gpr, meas_gpr)
                    y_true = te["soh"].values.astype(float)
                    met = regression_metrics(y_true, pred)
                    rows.append({
                        "experiment": "ApproxPaper-GPR-EKF",
                        "window": window,
                        "test_cell": test_cell,
                        "model": "ApproxPaper-GPR-EKF",
                        "exclude_outliers": exclude_outliers,
                        "n_train": len(train_df),
                        "n_test": len(te),
                        "n_features": 1,
                        **met,
                    })
                except Exception as e:
                    print(f"FAILED ApproxPaper-GPR-EKF {window}, {test_cell}: {e}")
    return pd.DataFrame(rows)

approx_paper_gpr_ekf_results = evaluate_approx_paper_gpr_ekf(df)
approx_paper_gpr_ekf_results.to_csv(os.path.join(RESULT_DIR, "approx_paper_gpr_ekf_leave_one_cell_results.csv"), index=False)
display(approx_paper_gpr_ekf_results.head())

approx_paper_gpr_ekf_summary = (
    approx_paper_gpr_ekf_results
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
approx_paper_gpr_ekf_summary.to_csv(os.path.join(RESULT_DIR, "approx_paper_gpr_ekf_summary_mean_std.csv"), index=False)
display(approx_paper_gpr_ekf_summary)

In [ ]:
# ------------------------------------------------------------
# 9) DEEP SEQUENCE MODELS: LSTM, GRU, TCN, TRANSFORMER
# ------------------------------------------------------------
if RUN_DEEP_MODELS:
    import random
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Deep learning device:", DEVICE)

    def set_torch_seed(seed=RANDOM_STATE):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    class SeqDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    def build_sequences(dataframe, feature_cols, seq_len=5):
        X_list, y_list, meta = [], [], []

        for cell, g in dataframe.sort_values(["cell", "cycle_number"]).groupby("cell"):
            g = g.dropna(subset=["soh"]).copy()

            if len(g) < seq_len:
                continue

            X = g[feature_cols].values.astype(float)
            y = g["soh"].values.astype(float)
            cycles = g["cycle_number"].values

            for i in range(seq_len - 1, len(g)):
                X_list.append(X[i - seq_len + 1:i + 1])
                y_list.append(y[i])
                meta.append({
                    "cell": cell,
                    "cycle_number": cycles[i]
                })

        return (
            np.asarray(X_list, dtype=float),
            np.asarray(y_list, dtype=float),
            pd.DataFrame(meta)
        )

    # ------------------------------------------------------------
    # SAFE PREPROCESSING
    # ------------------------------------------------------------
    def preprocess_train_test_for_deep(train_df, test_df, feature_cols):
        """
        Deep learning modelleri için güvenli preprocessing.

        Neden gerekli?
        Dataset 3'te bazı feature kolonları tamamen NaN olabiliyor.
        SimpleImputer bazı sürümlerde tamamen NaN kolonları düşürebiliyor.
        Bu da 'Columns must be same length as key' hatasına neden oluyor.

        Çözüm:
        - Feature isimlerini tekilleştirir.
        - inf değerleri NaN yapar.
        - Train içinde tamamen NaN olan kolonları 0 ile doldurur.
        - Median imputation ve StandardScaler uygular.
        - Kolon sayısını koruyarak DataFrame'e geri yazar.
        """

        # Olası duplicate feature isimlerini kaldır
        feature_cols = list(dict.fromkeys(feature_cols))

        X_train_df = train_df[feature_cols].copy()
        X_test_df = test_df[feature_cols].copy()

        # Sayısal dönüşüm
        for col in feature_cols:
            X_train_df[col] = pd.to_numeric(X_train_df[col], errors="coerce")
            X_test_df[col] = pd.to_numeric(X_test_df[col], errors="coerce")

        X_train_df = X_train_df.replace([np.inf, -np.inf], np.nan)
        X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan)

        # Train tarafında tamamen NaN olan kolonlar
        all_nan_cols = X_train_df.columns[X_train_df.isna().all()].tolist()

        if len(all_nan_cols) > 0:
            print("All-NaN columns filled with 0 for deep models:", all_nan_cols)

            X_train_df.loc[:, all_nan_cols] = 0.0

            for col in all_nan_cols:
                if col in X_test_df.columns:
                    X_test_df.loc[:, col] = 0.0

        # Test tarafında tamamen NaN olup train tarafında olmayan kolonlar varsa
        # imputer train median ile doldurur. Eğer testte de problem kalırsa 0'a çekiyoruz.
        X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan)

        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_imp = imputer.fit_transform(X_train_df)
        X_test_imp = imputer.transform(X_test_df)

        # Güvenlik: sklearn sürümü tüm-NaN kolon düşürdüyse yakala
        if X_train_imp.shape[1] != len(feature_cols):
            raise RuntimeError(
                f"Imputer feature size mismatch. "
                f"Expected {len(feature_cols)}, got {X_train_imp.shape[1]}. "
                f"All-NaN columns were: {all_nan_cols}"
            )

        X_train_scaled = scaler.fit_transform(X_train_imp)
        X_test_scaled = scaler.transform(X_test_imp)

        if X_train_scaled.shape[1] != len(feature_cols):
            raise RuntimeError(
                f"Scaler feature size mismatch. "
                f"Expected {len(feature_cols)}, got {X_train_scaled.shape[1]}"
            )

        train_df_proc = train_df.copy()
        test_df_proc = test_df.copy()

        train_df_proc.loc[:, feature_cols] = X_train_scaled
        test_df_proc.loc[:, feature_cols] = X_test_scaled

        return train_df_proc, test_df_proc, feature_cols

    class LSTMRegressor(nn.Module):
        def __init__(self, n_features, hidden=32, layers=1, dropout=0.1):
            super().__init__()
            self.rnn = nn.LSTM(
                n_features,
                hidden,
                num_layers=layers,
                batch_first=True,
                dropout=dropout if layers > 1 else 0.0
            )
            self.head = nn.Sequential(
                nn.Linear(hidden, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(out[:, -1, :])

    class GRURegressor(nn.Module):
        def __init__(self, n_features, hidden=32, layers=1, dropout=0.1):
            super().__init__()
            self.rnn = nn.GRU(
                n_features,
                hidden,
                num_layers=layers,
                batch_first=True,
                dropout=dropout if layers > 1 else 0.0
            )
            self.head = nn.Sequential(
                nn.Linear(hidden, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(out[:, -1, :])

    class TCNRegressor(nn.Module):
        def __init__(self, n_features, hidden=32, dropout=0.1):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(n_features, hidden, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            self.head = nn.Sequential(
                nn.Linear(hidden, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            # x: batch, seq, feat -> batch, feat, seq
            z = x.permute(0, 2, 1)
            z = self.net(z)
            z = z[:, :, -1]
            return self.head(z)

    class TransformerRegressor(nn.Module):
        def __init__(self, n_features, d_model=32, nhead=4, num_layers=2, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(n_features, d_model)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=64,
                dropout=dropout,
                batch_first=True,
                activation="gelu"
            )
            self.encoder = nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
            self.head = nn.Sequential(
                nn.Linear(d_model, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            z = self.input_proj(x)
            z = self.encoder(z)
            return self.head(z[:, -1, :])

    def train_torch_regressor(model, X_train, y_train, X_val, y_val, epochs=80):
        model = model.to(DEVICE)

        train_loader = DataLoader(
            SeqDataset(X_train, y_train),
            batch_size=DEEP_BATCH_SIZE,
            shuffle=True
        )

        val_X = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        val_y = torch.tensor(y_val, dtype=torch.float32).view(-1, 1).to(DEVICE)

        opt = torch.optim.AdamW(
            model.parameters(),
            lr=DEEP_LR,
            weight_decay=1e-4
        )

        loss_fn = nn.MSELoss()
        best_state = None
        best_val = float("inf")
        wait = 0

        for ep in range(epochs):
            model.train()

            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)

                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            model.eval()

            with torch.no_grad():
                val_loss = float(loss_fn(model(val_X), val_y).item())

            if val_loss < best_val - 1e-8:
                best_val = val_loss
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                wait = 0
            else:
                wait += 1

                if wait >= DEEP_PATIENCE:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        return model

    def predict_torch(model, X):
        model.eval()

        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
            pred = model(X_t).detach().cpu().numpy().reshape(-1)

        return pred

    def evaluate_deep_models(dataframe, windows=WINDOWS, exclude_outliers=True):
        rows = []

        all_cells = sorted(dataframe["cell"].unique())
        data_eval = dataframe.copy()

        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        model_builders = {
            "LSTM": lambda nf: LSTMRegressor(nf),
            "GRU": lambda nf: GRURegressor(nf),
            "TCN": lambda nf: TCNRegressor(nf),
            "TransformerEncoder": lambda nf: TransformerRegressor(nf),
        }

        for window in windows:
            feature_cols = get_window_feature_cols(
                data_eval,
                window,
                USE_CYCLE_NUMBER,
                USE_GLOBAL_TEMPERATURE
            )

            # Duplicate feature isimlerini kaldır
            feature_cols = list(dict.fromkeys(feature_cols))

            print(f"\nDeep models - window={window}, n_features={len(feature_cols)}")

            for test_cell in all_cells:
                train_df = data_eval[data_eval["cell"] != test_cell].copy()
                test_df = data_eval[data_eval["cell"] == test_cell].copy()

                if len(train_df) == 0 or len(test_df) == 0:
                    continue

                try:
                    train_df_proc, test_df_proc, feature_cols_safe = preprocess_train_test_for_deep(
                        train_df=train_df,
                        test_df=test_df,
                        feature_cols=feature_cols
                    )
                except Exception as e:
                    print(f"FAILED preprocessing deep, {window}, {test_cell}: {e}")
                    continue

                # Split train cells into train/val by taking one validation cell from training set
                train_cells = sorted(train_df_proc["cell"].unique())

                if len(train_cells) < 2:
                    continue

                val_cell = train_cells[-1]

                proper_train = train_df_proc[train_df_proc["cell"] != val_cell].copy()
                val_df = train_df_proc[train_df_proc["cell"] == val_cell].copy()

                X_tr, y_tr, _ = build_sequences(
                    proper_train,
                    feature_cols_safe,
                    seq_len=DEEP_SEQ_LEN
                )

                X_val, y_val, _ = build_sequences(
                    val_df,
                    feature_cols_safe,
                    seq_len=DEEP_SEQ_LEN
                )

                X_te, y_te, _ = build_sequences(
                    test_df_proc,
                    feature_cols_safe,
                    seq_len=DEEP_SEQ_LEN
                )

                if len(X_tr) < 20 or len(X_val) < 5 or len(X_te) < 5:
                    print(
                        f"SKIPPED deep {window}, {test_cell}: "
                        f"not enough sequences "
                        f"(train={len(X_tr)}, val={len(X_val)}, test={len(X_te)})"
                    )
                    continue

                for model_name, builder in model_builders.items():
                    try:
                        set_torch_seed(RANDOM_STATE)

                        model = builder(len(feature_cols_safe))

                        model = train_torch_regressor(
                            model,
                            X_tr,
                            y_tr,
                            X_val,
                            y_val,
                            epochs=DEEP_EPOCHS
                        )

                        pred = predict_torch(model, X_te)
                        pred = np.clip(pred, 0.5, 1.05)

                        met = regression_metrics(y_te, pred)

                        rows.append({
                            "experiment": "MultiHI-DeepSequence",
                            "window": window,
                            "test_cell": test_cell,
                            "model": model_name,
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(X_tr),
                            "n_val": len(X_val),
                            "n_test": len(X_te),
                            "n_features": len(feature_cols_safe),
                            "seq_len": DEEP_SEQ_LEN,
                            **met,
                        })

                        print(
                            f"Deep OK | {window} | {test_cell} | {model_name} | "
                            f"RMSE={met['RMSE']:.5f} | MAE={met['MAE']:.5f} | R2={met['R2']:.5f}"
                        )

                    except Exception as e:
                        print(f"FAILED deep {model_name}, {window}, {test_cell}: {e}")

        return pd.DataFrame(rows)

    deep_results = evaluate_deep_models(
        df,
        windows=WINDOWS,
        exclude_outliers=True
    )

    deep_results.to_csv(
        os.path.join(RESULT_DIR, "multi_hi_deep_sequence_leave_one_cell_results.csv"),
        index=False
    )

    display(deep_results.head())

    if not deep_results.empty:
        deep_summary = (
            deep_results
            .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
            .agg(["mean", "std"])
            .reset_index()
        )
    else:
        deep_summary = pd.DataFrame()

    deep_summary.to_csv(
        os.path.join(RESULT_DIR, "multi_hi_deep_sequence_summary_mean_std.csv"),
        index=False
    )

    display(deep_summary)

else:
    deep_results = pd.DataFrame()
    deep_summary = pd.DataFrame()

In [ ]:
# ------------------------------------------------------------
# 10) COMBINE ALL RESULTS AND CREATE COMPARATIVE TABLES
# ------------------------------------------------------------
all_results_parts = [
    single_hi_ekf_results_v2,
    approx_paper_gpr_ekf_results,
    multi_hi_tabular_results,
]
if RUN_DEEP_MODELS and not deep_results.empty:
    all_results_parts.append(deep_results)

all_model_results = pd.concat(all_results_parts, ignore_index=True)
all_model_results.to_csv(os.path.join(RESULT_DIR, "all_leave_one_cell_model_results.csv"), index=False)

all_summary = (
    all_model_results
    .groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
all_summary.to_csv(os.path.join(RESULT_DIR, "all_model_summary_mean_std.csv"), index=False)
print("\nALL MODEL SUMMARY:")
display(all_summary)

# Best model per window
best_per_window = (
    all_model_results.groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]]
    .mean()
    .reset_index()
    .sort_values(["exclude_outliers", "window", "RMSE"])
)
best_per_window.to_csv(os.path.join(RESULT_DIR, "best_models_ranked_by_window.csv"), index=False)
display(best_per_window.groupby(["exclude_outliers", "window"]).head(10))

# Single-HI vs best Multi-HI comparison
single_avg = (
    single_hi_ekf_results_v2
    .groupby(["exclude_outliers", "window"])[["RMSE", "MAE", "R2"]]
    .mean().reset_index()
    .rename(columns={"RMSE": "single_hi_rmse", "MAE": "single_hi_mae", "R2": "single_hi_r2"})
)

multi_candidates = all_model_results[all_model_results["experiment"].isin(["MultiHI-Tabular", "MultiHI-DeepSequence"])].copy()
multi_avg = (
    multi_candidates
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .mean().reset_index()
)
best_multi = multi_avg.sort_values(["exclude_outliers", "window", "RMSE"]).groupby(["exclude_outliers", "window"]).head(1)
best_multi = best_multi.rename(columns={"model": "best_multi_model", "RMSE": "best_multi_rmse", "MAE": "best_multi_mae", "R2": "best_multi_r2"})

single_vs_multi = pd.merge(single_avg, best_multi, on=["exclude_outliers", "window"], how="left")
single_vs_multi["rmse_improvement_percent"] = 100.0 * (single_vs_multi["single_hi_rmse"] - single_vs_multi["best_multi_rmse"]) / single_vs_multi["single_hi_rmse"]
single_vs_multi.to_csv(os.path.join(RESULT_DIR, "single_hi_vs_best_multi_hi_summary.csv"), index=False)
print("\nSingle-HI vs Best Multi-HI:")
display(single_vs_multi)

In [ ]:
# ------------------------------------------------------------
# 10B) PROPOSED MODEL: MAW-GME
# Missing-Aware Window-Gated Multi-HI Ensemble
# ------------------------------------------------------------
# Bu hücre sadece önerilen MAW-GME modelini ekler.
# Mantık:
# - Her partial charging window için uzman modeller eğitilir.
# - Her uzman modelin güvenilirliği, yalnızca training hücreleri içinde yapılan inner leave-one-cell CV RMSE ile hesaplanır.
# - Test örneğinde ilgili window feature'larının ne kadar mevcut olduğuna bakılır.
# - Daha düşük inner RMSE ve daha yüksek feature availability daha yüksek ağırlık alır.
# - Nihai SoH tahmini, window/model uzmanlarının ağırlıklı ortalamasıdır.

MAWGME_WINDOW_NAME = "adaptive_all_windows"
MAWGME_EXPERT_CANDIDATES = ["Ridge", "ExtraTrees", "XGBoost", "LightGBM", "CatBoost"]

def _mawgme_feature_availability(X, hi_cols):
    """
    Her test örneği için ilgili HI kolonlarının mevcut olma oranını hesaplar.
    cycle_number ve global sıcaklık feature'ları bu hesaba katılmaz.
    """
    if len(hi_cols) == 0:
        return np.ones(len(X), dtype=float)
    return X[hi_cols].notna().mean(axis=1).values.astype(float)

def _mawgme_inner_cell_cv_rmse(train_df, feature_cols, base_model):
    """
    Gating için expert güvenilirliği.
    Sadece training hücreleri içinde leave-one-cell-out yapılır.
    Test hücresi bu hesapta kullanılmaz, böylece leakage oluşmaz.
    """
    cells = sorted(train_df["cell"].unique())
    y_all, pred_all = [], []

    for val_cell in cells:
        tr = train_df[train_df["cell"] != val_cell].copy()
        va = train_df[train_df["cell"] == val_cell].copy()

        if len(tr) < 20 or len(va) < 5:
            continue

        try:
            model = clone(base_model)
            model.fit(tr[feature_cols], tr["soh"].values.astype(float))
            pred = model.predict(va[feature_cols])
            pred = np.clip(pred, 0.5, 1.05)
            y_all.extend(va["soh"].values.astype(float).tolist())
            pred_all.extend(pred.tolist())
        except Exception:
            continue

    if len(y_all) < 10:
        return 0.05

    return max(rmse(np.asarray(y_all), np.asarray(pred_all)), 1e-6)

def evaluate_maw_gme(dataframe, windows=WINDOWS, exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS):
    """
    MAW-GME leave-one-cell-out değerlendirmesi.
    Çıktı formatı mevcut all_model_results ile uyumludur.
    """
    rows = []
    base_models = make_tabular_models()
    expert_names = [m for m in MAWGME_EXPERT_CANDIDATES if m in base_models]
    all_cells = sorted(dataframe["cell"].unique())

    if len(expert_names) == 0:
        print("MAW-GME skipped: no available expert models.")
        return pd.DataFrame()

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()

        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for test_cell in all_cells:
            train_all = data_eval[data_eval["cell"] != test_cell].copy()
            test_all = data_eval[data_eval["cell"] == test_cell].copy()

            if len(train_all) < 20 or len(test_all) < 5:
                continue

            y_test = test_all["soh"].values.astype(float)
            expert_predictions = []
            expert_weights = []
            expert_info = []

            for window in windows:
                feature_cols = get_window_feature_cols(
                    data_eval,
                    window,
                    USE_CYCLE_NUMBER,
                    USE_GLOBAL_TEMPERATURE
                )

                hi_cols = [
                    c for c in feature_cols
                    if c not in ["cycle_number", "temp_mean", "temp_max"]
                ]

                for expert_name in expert_names:
                    base_model = base_models[expert_name]

                    try:
                        inner_rmse = _mawgme_inner_cell_cv_rmse(
                            train_all[["cell", "soh"] + feature_cols].copy(),
                            feature_cols,
                            base_model
                        )

                        model = clone(base_model)
                        model.fit(train_all[feature_cols], train_all["soh"].values.astype(float))

                        X_test = test_all[feature_cols].copy()
                        pred = model.predict(X_test)
                        pred = np.clip(pred, 0.5, 1.05)

                        availability = _mawgme_feature_availability(X_test, hi_cols)

                        # İç CV hatası düşük olan expert daha güvenilir kabul edilir.
                        # availability düşükse o expert'in test örneği için ağırlığı düşer.
                        reliability = 1.0 / ((inner_rmse + 1e-6) ** 2)
                        sample_weight = availability * reliability

                        expert_predictions.append(pred)
                        expert_weights.append(sample_weight)
                        expert_info.append({
                            "window": window,
                            "expert": expert_name,
                            "inner_rmse": inner_rmse,
                        })

                    except Exception as e:
                        print(f"MAW-GME expert failed: test_cell={test_cell}, window={window}, expert={expert_name}, error={e}")

            if len(expert_predictions) == 0:
                continue

            pred_matrix = np.vstack(expert_predictions).T
            weight_matrix = np.vstack(expert_weights).T

            weight_sum = weight_matrix.sum(axis=1, keepdims=True)
            bad_rows = weight_sum.squeeze() <= 0

            if np.any(bad_rows):
                weight_matrix[bad_rows, :] = 1.0
                weight_sum = weight_matrix.sum(axis=1, keepdims=True)

            norm_weights = weight_matrix / weight_sum
            final_pred = np.sum(pred_matrix * norm_weights, axis=1)
            final_pred = np.clip(final_pred, 0.5, 1.05)

            met = regression_metrics(y_test, final_pred)

            rows.append({
                "experiment": "MAW-GME",
                "window": MAWGME_WINDOW_NAME,
                "test_cell": test_cell,
                "model": "MAW-GME",
                "exclude_outliers": exclude_outliers,
                "n_features": int(sum(len(get_window_feature_cols(data_eval, w, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)) for w in windows)),
                "n_experts": len(expert_predictions),
                **met,
            })

    return pd.DataFrame(rows)

mawgme_results = evaluate_maw_gme(df)
mawgme_results.to_csv(os.path.join(RESULT_DIR, "maw_gme_leave_one_cell_results.csv"), index=False)

if not mawgme_results.empty:
    # Hücre tekrar çalıştırılırsa eski MAW-GME satırlarını çıkarıp yenilerini ekle.
    if "all_model_results" in globals() and not all_model_results.empty:
        all_model_results = all_model_results[all_model_results["experiment"] != "MAW-GME"].copy()
        all_model_results = pd.concat([all_model_results, mawgme_results], ignore_index=True)
    else:
        all_model_results = mawgme_results.copy()

    all_model_results.to_csv(os.path.join(RESULT_DIR, "all_leave_one_cell_model_results.csv"), index=False)

    mawgme_summary = (
        mawgme_results
        .groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]]
        .agg(["mean", "std"])
        .reset_index()
    )
    mawgme_summary.to_csv(os.path.join(RESULT_DIR, "maw_gme_summary_mean_std.csv"), index=False)

    # Genel özet tablolarını MAW-GME dahil olacak şekilde güncelle.
    all_summary = (
        all_model_results
        .groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]]
        .agg(["mean", "std"])
        .reset_index()
    )
    all_summary.to_csv(os.path.join(RESULT_DIR, "all_model_summary_mean_std.csv"), index=False)

    best_per_window = (
        all_model_results.groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]]
        .mean()
        .reset_index()
        .sort_values(["exclude_outliers", "window", "RMSE"])
    )
    best_per_window.to_csv(os.path.join(RESULT_DIR, "best_models_ranked_by_window.csv"), index=False)

    print("\nMAW-GME RESULTS:")
    display(mawgme_results)

    print("\nMAW-GME SUMMARY:")
    display(mawgme_summary)
else:
    print("MAW-GME did not produce any result.")

In [ ]:
# ------------------------------------------------------------
# 11) STRUCTURED PARTIAL-WINDOW AND FEATURE-GROUP ABLATION
#     + MAW-GME INTEGRATION FOR DATASET 3
# ------------------------------------------------------------
# Random feature deletion is not used in this version.
# Instead, we evaluate realistic partial-charging availability scenarios:
#   1) all windows available
#   2) only full window
#   3) only medium window
#   4) only narrow window
#   5) only very narrow window
#   6) all windows but Pmax-based features removed
#   7) all windows but ICA-shape features removed
#
# In this version, MAW-GME is also evaluated under the same 7 scenarios.

STRUCTURED_SCENARIOS = [
    "all_windows_available",
    "only_full_window",
    "only_medium_window",
    "only_narrow_window",
    "only_very_narrow_window",
    "pmax_features_missing",
    "ica_shape_features_missing",
]

STRUCTURED_SCENARIO_DESCRIPTIONS = {
    "all_windows_available": "All voltage-window features are available.",
    "only_full_window": "Only full-window features are available.",
    "only_medium_window": "Only medium-window features are available.",
    "only_narrow_window": "Only narrow-window features are available.",
    "only_very_narrow_window": "Only very-narrow-window features are available.",
    "pmax_features_missing": "All windows are available, but pmax, pmax_voltage, and pmax_norm features are removed.",
    "ica_shape_features_missing": "All windows are available, but ICA-shape features such as ic_std, peak_width, and DVA features are removed.",
}


def _first_window_by_prefix(windows, prefix):
    """
    Dataset 3 için esnek pencere seçimi.
    Örneğin:
        full -> full_27_36
        medium -> medium_32_35
        narrow -> narrow_33_345
        very_narrow -> very_narrow_335_340
    """
    candidates = [w for w in windows if w.startswith(prefix)]

    if len(candidates) == 0:
        raise ValueError(
            f"No window starts with prefix={prefix}. Available windows: {windows}"
        )

    return candidates[0]


def unique_preserve_order(cols):
    return list(dict.fromkeys(cols))


def get_all_window_feature_cols_for_structured(dataframe, windows=WINDOWS):
    """
    Union of leakage-clean feature columns across all windows.
    """
    cols = []

    for w in windows:
        cols.extend(
            get_window_feature_cols(
                dataframe,
                w,
                USE_CYCLE_NUMBER,
                USE_GLOBAL_TEMPERATURE
            )
        )

    return unique_preserve_order(cols)


def apply_structured_feature_filter(cols, scenario):
    """
    Senaryoya göre feature grubu çıkarımı.
    """
    cols = unique_preserve_order(cols)

    if scenario == "pmax_features_missing":
        cols = [
            c for c in cols
            if "pmax" not in c.lower()
        ]

    elif scenario == "ica_shape_features_missing":
        shape_tokens = [
            "ic_std",
            "peak_width",
            "dva",
        ]

        cols = [
            c for c in cols
            if not any(tok in c.lower() for tok in shape_tokens)
        ]

    return unique_preserve_order(cols)


def remove_unusable_feature_cols(dataframe, feature_cols, scenario_name=""):
    """
    Remove duplicate, missing, and completely empty columns.

    Dataset 3 RPT files do not contain temperature measurements in the selected files,
    so temperature columns may be entirely NaN. Removing them is cleaner than imputing
    non-existent measurements.
    """
    feature_cols = unique_preserve_order(feature_cols)
    feature_cols = [c for c in feature_cols if c in dataframe.columns]

    all_nan_cols = [c for c in feature_cols if dataframe[c].isna().all()]

    if len(all_nan_cols) > 0:
        print(f"[{scenario_name}] removed all-NaN feature columns:", all_nan_cols)

    feature_cols = [c for c in feature_cols if c not in all_nan_cols]

    # Final leakage check
    bad = [
        c for c in feature_cols
        if c in LEAKAGE_COLUMNS_EXACT
        or c.startswith("soh")
        or c in ["capacity_mAh", "capacity_Ah"]
    ]

    if bad:
        raise ValueError(
            f"Leakage columns found in structured scenario {scenario_name}: {bad}"
        )

    feature_cols = unique_preserve_order(feature_cols)

    if len(feature_cols) == 0:
        raise ValueError(f"No usable features remain for scenario={scenario_name}")

    return feature_cols, all_nan_cols


def get_structured_scenario_feature_cols(dataframe, scenario, windows=WINDOWS):
    """
    Build feature sets for 7 structured partial-window / feature-group scenarios.
    These scenarios emulate realistic partial charging more directly than random feature deletion.
    """
    if scenario == "all_windows_available":
        cols = get_all_window_feature_cols_for_structured(dataframe, windows)

    elif scenario == "only_full_window":
        w = _first_window_by_prefix(windows, "full")
        cols = get_window_feature_cols(
            dataframe,
            w,
            USE_CYCLE_NUMBER,
            USE_GLOBAL_TEMPERATURE
        )

    elif scenario == "only_medium_window":
        w = _first_window_by_prefix(windows, "medium")
        cols = get_window_feature_cols(
            dataframe,
            w,
            USE_CYCLE_NUMBER,
            USE_GLOBAL_TEMPERATURE
        )

    elif scenario == "only_narrow_window":
        # Important: this intentionally selects the regular narrow window,
        # not the paper-specific window.
        w = _first_window_by_prefix(windows, "narrow")
        cols = get_window_feature_cols(
            dataframe,
            w,
            USE_CYCLE_NUMBER,
            USE_GLOBAL_TEMPERATURE
        )

    elif scenario == "only_very_narrow_window":
        w = _first_window_by_prefix(windows, "very_narrow")
        cols = get_window_feature_cols(
            dataframe,
            w,
            USE_CYCLE_NUMBER,
            USE_GLOBAL_TEMPERATURE
        )

    elif scenario in ["pmax_features_missing", "ica_shape_features_missing"]:
        cols = get_all_window_feature_cols_for_structured(dataframe, windows)
        cols = apply_structured_feature_filter(cols, scenario)

    else:
        raise ValueError(f"Unknown structured scenario: {scenario}")

    cols, removed_all_nan = remove_unusable_feature_cols(
        dataframe,
        cols,
        scenario_name=scenario
    )

    return cols, removed_all_nan


def get_maw_gme_windows_for_scenario(scenario, windows=WINDOWS):
    """
    MAW-GME için senaryoya göre hangi pencere uzmanlarının kullanılacağını belirler.
    """
    if scenario == "all_windows_available":
        return list(windows)

    if scenario == "only_full_window":
        return [_first_window_by_prefix(windows, "full")]

    if scenario == "only_medium_window":
        return [_first_window_by_prefix(windows, "medium")]

    if scenario == "only_narrow_window":
        return [_first_window_by_prefix(windows, "narrow")]

    if scenario == "only_very_narrow_window":
        return [_first_window_by_prefix(windows, "very_narrow")]

    if scenario in ["pmax_features_missing", "ica_shape_features_missing"]:
        return list(windows)

    raise ValueError(f"Unknown structured scenario for MAW-GME: {scenario}")


def build_maw_gme_expert_specs(dataframe, scenario, base_models, selected_names, windows=WINDOWS):
    """
    MAW-GME uzmanlarını oluşturur.
    Her uzman = bir pencere + bir tabular model.
    """
    expert_specs = []
    expert_windows = get_maw_gme_windows_for_scenario(scenario, windows)

    for w in expert_windows:
        cols = get_window_feature_cols(
            dataframe,
            w,
            USE_CYCLE_NUMBER,
            USE_GLOBAL_TEMPERATURE
        )

        cols = apply_structured_feature_filter(cols, scenario)

        cols, removed_all_nan = remove_unusable_feature_cols(
            dataframe,
            cols,
            scenario_name=f"MAW-GME::{scenario}::{w}"
        )

        for model_name in selected_names:
            if model_name in base_models:
                expert_specs.append({
                    "window": w,
                    "model_name": model_name,
                    "feature_cols": cols,
                    "n_features": len(cols),
                })

    if len(expert_specs) == 0:
        raise ValueError(f"No MAW-GME experts could be built for scenario={scenario}")

    return expert_specs


def fit_predict_structured_maw_gme(train_df, test_df, expert_specs, base_models):
    """
    Structured senaryolar için MAW-GME tahmini.

    Mantık:
    - Train hücrelerinden bir tanesi validasyon hücresi olarak ayrılır.
    - Her expert için validasyon RMSE hesaplanır.
    - Daha düşük validasyon hatasına sahip expert daha yüksek ağırlık alır.
    - Son tahmin, expert tahminlerinin ağırlıklı ortalamasıdır.

    Not:
    Bu yapı test hücresini ağırlık öğreniminde kullanmaz; leakage oluşturmaz.
    """
    train_cells = sorted(train_df["cell"].unique())

    if len(train_cells) < 2:
        raise ValueError(
            "MAW-GME needs at least two training cells for internal validation."
        )

    val_cell = train_cells[-1]

    fit_df = train_df[train_df["cell"] != val_cell].copy()
    val_df = train_df[train_df["cell"] == val_cell].copy()

    y_fit = fit_df["soh"].values.astype(float)
    y_val = val_df["soh"].values.astype(float)
    y_train_full = train_df["soh"].values.astype(float)

    expert_preds = []
    expert_errors = []
    expert_info = []

    for spec in expert_specs:
        model_name = spec["model_name"]
        cols = spec["feature_cols"]

        if len(cols) == 0:
            continue

        try:
            # 1) Validation model
            val_model = clone(base_models[model_name])
            val_model.fit(fit_df[cols], y_fit)

            val_pred = val_model.predict(val_df[cols])
            val_pred = np.clip(val_pred, 0.5, 1.05)

            val_rmse = float(
                np.sqrt(np.mean((y_val - val_pred) ** 2))
            )

            if not np.isfinite(val_rmse):
                continue

            # 2) Final model, full train üzerinde tekrar eğitilir
            final_model = clone(base_models[model_name])
            final_model.fit(train_df[cols], y_train_full)

            test_pred = final_model.predict(test_df[cols])
            test_pred = np.clip(test_pred, 0.5, 1.05)

            expert_preds.append(test_pred)
            expert_errors.append(val_rmse)
            expert_info.append({
                "window": spec["window"],
                "model_name": model_name,
                "n_features": len(cols),
                "val_rmse": val_rmse,
            })

        except Exception as e:
            print(
                f"FAILED MAW-GME expert window={spec['window']}, "
                f"model={model_name}: {e}"
            )

    if len(expert_preds) == 0:
        raise ValueError("No valid MAW-GME expert predictions were produced.")

    expert_preds = np.vstack(expert_preds)
    expert_errors = np.asarray(expert_errors, dtype=float)

    # Daha düşük validation RMSE -> daha yüksek ağırlık
    raw_weights = 1.0 / (expert_errors + 1e-8)
    weights = raw_weights / raw_weights.sum()

    final_pred = np.sum(expert_preds * weights[:, None], axis=0)
    final_pred = np.clip(final_pred, 0.5, 1.05)

    for i, info in enumerate(expert_info):
        info["weight"] = float(weights[i])

    return final_pred, expert_info


def evaluate_structured_window_ablation(
    dataframe,
    scenarios=STRUCTURED_SCENARIOS,
    exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS
):
    """
    Leave-one-cell-out evaluation under structured partial-window availability scenarios.
    This replaces random missing-HI as the main robustness experiment.

    This version evaluates both:
    - standard tabular Multi-HI models
    - proposed MAW-GME model
    """
    rows = []
    feature_rows = []
    maw_gme_weight_rows = []

    base_models = make_tabular_models()

    # Compact but representative model set to reduce training time.
    selected_names = [
        name for name in
        ["Ridge", "ExtraTrees", "RandomForest", "HistGB", "XGBoost", "LightGBM", "CatBoost"]
        if name in base_models
    ]

    all_cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()

        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for scenario in scenarios:
            try:
                feature_cols, removed_all_nan = get_structured_scenario_feature_cols(
                    data_eval,
                    scenario,
                    WINDOWS
                )
            except Exception as e:
                print(f"SKIPPED structured scenario {scenario}: {e}")
                continue

            for f in feature_cols:
                feature_rows.append({
                    "scenario": scenario,
                    "description": STRUCTURED_SCENARIO_DESCRIPTIONS.get(scenario, ""),
                    "exclude_outliers": exclude_outliers,
                    "feature": f,
                    "n_features": len(feature_cols),
                })

            print(
                f"\nStructured scenario={scenario} | "
                f"exclude_outliers={exclude_outliers} | "
                f"n_features={len(feature_cols)}"
            )

            # ------------------------------------------------------------
            # A) Standard tabular models
            # ------------------------------------------------------------
            for model_name in selected_names:
                for test_cell in all_cells:
                    train_df = data_eval[data_eval["cell"] != test_cell].copy()
                    test_df = data_eval[data_eval["cell"] == test_cell].copy()

                    if len(train_df) < 20 or len(test_df) < 5:
                        continue

                    X_train = train_df[feature_cols]
                    y_train = train_df["soh"].values.astype(float)
                    X_test = test_df[feature_cols]
                    y_test = test_df["soh"].values.astype(float)

                    try:
                        model = clone(base_models[model_name])
                        model.fit(X_train, y_train)

                        pred = model.predict(X_test)
                        pred = np.clip(pred, 0.5, 1.05)

                        met = regression_metrics(y_test, pred)

                        rows.append({
                            "experiment": "StructuredWindowAblation",
                            "scenario": scenario,
                            "scenario_description": STRUCTURED_SCENARIO_DESCRIPTIONS.get(scenario, ""),
                            "test_cell": test_cell,
                            "model": model_name,
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(train_df),
                            "n_test": len(test_df),
                            "n_features": len(feature_cols),
                            "n_experts": np.nan,
                            **met,
                        })

                    except Exception as e:
                        print(
                            f"FAILED structured {scenario}, {model_name}, {test_cell}: {e}"
                        )

            # ------------------------------------------------------------
            # B) Proposed MAW-GME under the same structured scenario
            # ------------------------------------------------------------
            try:
                expert_specs = build_maw_gme_expert_specs(
                    dataframe=data_eval,
                    scenario=scenario,
                    base_models=base_models,
                    selected_names=selected_names,
                    windows=WINDOWS
                )
            except Exception as e:
                print(f"SKIPPED MAW-GME for scenario={scenario}: {e}")
                expert_specs = []

            if len(expert_specs) > 0:
                print(
                    f"MAW-GME scenario={scenario} | "
                    f"exclude_outliers={exclude_outliers} | "
                    f"n_experts={len(expert_specs)}"
                )

                for test_cell in all_cells:
                    train_df = data_eval[data_eval["cell"] != test_cell].copy()
                    test_df = data_eval[data_eval["cell"] == test_cell].copy()

                    if len(train_df) < 20 or len(test_df) < 5:
                        continue

                    y_test = test_df["soh"].values.astype(float)

                    try:
                        pred, expert_info = fit_predict_structured_maw_gme(
                            train_df=train_df,
                            test_df=test_df,
                            expert_specs=expert_specs,
                            base_models=base_models
                        )

                        met = regression_metrics(y_test, pred)

                        rows.append({
                            "experiment": "StructuredWindowAblation",
                            "scenario": scenario,
                            "scenario_description": STRUCTURED_SCENARIO_DESCRIPTIONS.get(scenario, ""),
                            "test_cell": test_cell,
                            "model": "MAW-GME",
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(train_df),
                            "n_test": len(test_df),
                            "n_features": len(feature_cols),
                            "n_experts": len(expert_specs),
                            **met,
                        })

                        for info in expert_info:
                            maw_gme_weight_rows.append({
                                "scenario": scenario,
                                "scenario_description": STRUCTURED_SCENARIO_DESCRIPTIONS.get(scenario, ""),
                                "exclude_outliers": exclude_outliers,
                                "test_cell": test_cell,
                                **info,
                            })

                    except Exception as e:
                        print(
                            f"FAILED structured MAW-GME "
                            f"scenario={scenario}, test_cell={test_cell}: {e}"
                        )

    return (
        pd.DataFrame(rows),
        pd.DataFrame(feature_rows),
        pd.DataFrame(maw_gme_weight_rows)
    )


structured_window_ablation_results, structured_window_feature_sets, structured_maw_gme_weights = evaluate_structured_window_ablation(df)

structured_window_ablation_results.to_csv(
    os.path.join(RESULT_DIR, "structured_window_ablation_leave_one_cell_results.csv"),
    index=False
)

structured_window_feature_sets.to_csv(
    os.path.join(RESULT_DIR, "structured_window_ablation_feature_sets.csv"),
    index=False
)

structured_maw_gme_weights.to_csv(
    os.path.join(RESULT_DIR, "structured_window_ablation_maw_gme_expert_weights.csv"),
    index=False
)

if not structured_window_ablation_results.empty:
    structured_window_ablation_summary = (
        structured_window_ablation_results
        .groupby(["exclude_outliers", "scenario", "model"])[["RMSE", "MAE", "R2"]]
        .agg(["mean", "std"])
        .reset_index()
    )

    structured_window_ablation_best = (
        structured_window_ablation_results
        .groupby(["exclude_outliers", "scenario", "model"])[["RMSE", "MAE", "R2"]]
        .mean()
        .reset_index()
        .sort_values(["exclude_outliers", "scenario", "RMSE"])
    )

else:
    structured_window_ablation_summary = pd.DataFrame()
    structured_window_ablation_best = pd.DataFrame()

structured_window_ablation_summary.to_csv(
    os.path.join(RESULT_DIR, "structured_window_ablation_summary_mean_std.csv"),
    index=False
)

structured_window_ablation_best.to_csv(
    os.path.join(RESULT_DIR, "structured_window_ablation_best_models.csv"),
    index=False
)

print("\nStructured partial-window / feature-group ablation summary:")
display(structured_window_ablation_summary)

print("\nBest structured models per scenario:")
if not structured_window_ablation_best.empty:
    display(
        structured_window_ablation_best
        .groupby(["exclude_outliers", "scenario"])
        .head(8)
    )
else:
    display(structured_window_ablation_best)

print("\nStructured MAW-GME expert weights:")
display(structured_maw_gme_weights.head(50))

# Geriye dönük uyumluluk için eski random-missing değişken adlarını boş DataFrame yapıyoruz.
# Böylece notebook'un sonraki hücreleri eski isimleri beklese bile kırılmaz.
missing_hi_repeated = pd.DataFrame()
missing_hi_summary_mean_std = pd.DataFrame()

In [ ]:
# ------------------------------------------------------------
# 12) SHAP ANALYSIS AND PLOTS
# ------------------------------------------------------------
def train_model_for_shap(dataframe, window="medium_32_35", preferred_models=("XGBoost", "ExtraTrees", "LightGBM", "CatBoost", "RandomForest")):
    data_eval = dataframe.copy()
    if "soh_outlier" in data_eval.columns:
        data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

    feature_cols = get_window_feature_cols(data_eval, window, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
    X = data_eval[feature_cols]
    y = data_eval["soh"].values.astype(float)
    models = make_tabular_models()
    selected = None
    for name in preferred_models:
        if name in models:
            selected = name
            break
    if selected is None:
        selected = "ExtraTrees"
    model = clone(models[selected])
    model.fit(X, y)
    return selected, model, feature_cols, data_eval

try:
    import shap
    SHAP_WINDOW = "medium_32_35"
    shap_model_name, shap_pipeline, shap_feature_cols, shap_data = train_model_for_shap(df, SHAP_WINDOW)
    print("SHAP model:", shap_model_name, "window:", SHAP_WINDOW)

    # Transform features with pipeline preprocessing.
    X_raw = shap_data[shap_feature_cols]
    imputer = shap_pipeline.named_steps.get("imputer", None)
    scaler = shap_pipeline.named_steps.get("scaler", None)
    final_model = shap_pipeline.named_steps["model"]

    X_proc = X_raw.copy()
    if imputer is not None:
        X_proc = pd.DataFrame(imputer.transform(X_proc), columns=shap_feature_cols)
    if scaler is not None:
        X_proc = pd.DataFrame(scaler.transform(X_proc), columns=shap_feature_cols)

    # Sample for faster SHAP
    if len(X_proc) > 300:
        X_shap = X_proc.sample(300, random_state=RANDOM_STATE)
    else:
        X_shap = X_proc.copy()

    explainer = shap.Explainer(final_model, X_proc)
    shap_values = explainer(X_shap)

    shap_importance = pd.DataFrame({
        "feature": shap_feature_cols,
        "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False)
    shap_importance.to_csv(os.path.join(RESULT_DIR, "shap_importance_advanced.csv"), index=False)
    display(shap_importance.head(20))

    # Bar plot
    plt.figure(figsize=(9, 6))
    top = shap_importance.head(15).iloc[::-1]
    plt.barh(top["feature"], top["mean_abs_shap"])
    plt.xlabel("Mean |SHAP value|")
    plt.title(f"SHAP Feature Importance - {shap_model_name} - {SHAP_WINDOW}")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "shap_bar_plot.png"), dpi=300)
    plt.show()

    # Beeswarm plot
    shap.plots.beeswarm(shap_values, max_display=15, show=False)
    plt.title(f"SHAP Beeswarm - {shap_model_name} - {SHAP_WINDOW}")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "shap_beeswarm_plot.png"), dpi=300)
    plt.show()

    # Dependence/scatter for top feature
    top_feature = shap_importance.iloc[0]["feature"]
    shap.plots.scatter(shap_values[:, top_feature], show=False)
    plt.title(f"SHAP Dependence - {top_feature}")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "shap_dependence_top_feature.png"), dpi=300)
    plt.show()

except Exception as e:
    print("SHAP analysis failed:", e)

In [ ]:
# ------------------------------------------------------------
# 13) EXPORT EXCEL REPORT
# ------------------------------------------------------------

def make_excel_safe(df):
    """
    Excel'e yazmadan önce MultiIndex kolonları düzleştirir.
    Örneğin:
        ('RMSE', 'mean') -> RMSE_mean
        ('RMSE', 'std')  -> RMSE_std
    """
    if df is None:
        return pd.DataFrame()

    df = df.copy()

    # MultiIndex kolonları düzleştir
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x) for x in col if str(x) != ""]).strip("_")
            for col in df.columns.to_flat_index()
        ]
    else:
        df.columns = [str(c) for c in df.columns]

    # MultiIndex index varsa düzleştir
    if isinstance(df.index, pd.MultiIndex):
        df = df.reset_index()
    elif df.index.name is not None:
        df = df.reset_index()

    return df


def write_df_to_excel_if_available(writer, variable_name, sheet_name):
    """
    Değişken globals içinde varsa Excel'e yazar.
    Boş DataFrame ise de boş sheet oluşturur.
    Böylece bazı deneyler çalışmadığında export kısmı kırılmaz.
    """
    if variable_name in globals():
        obj = globals()[variable_name]

        if isinstance(obj, pd.DataFrame):
            make_excel_safe(obj).to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )


excel_path = os.path.join(RESULT_DIR, "advanced_soh_experiment_report.xlsx")

# Önceki başarısız Excel dosyası oluştuysa sil
if os.path.exists(excel_path):
    os.remove(excel_path)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    # --------------------------------------------------------
    # Main result sheets
    # --------------------------------------------------------
    write_df_to_excel_if_available(
        writer,
        "all_model_results",
        "all_cell_results"
    )

    write_df_to_excel_if_available(
        writer,
        "all_summary",
        "summary_mean_std"
    )

    write_df_to_excel_if_available(
        writer,
        "best_per_window",
        "best_models_ranked"
    )

    write_df_to_excel_if_available(
        writer,
        "single_vs_multi",
        "single_vs_multi"
    )

    # --------------------------------------------------------
    # MAW-GME standalone outputs
    # --------------------------------------------------------
    write_df_to_excel_if_available(
        writer,
        "maw_gme_results",
        "maw_gme_results"
    )

    write_df_to_excel_if_available(
        writer,
        "maw_gme_summary",
        "maw_gme_summary"
    )

    # --------------------------------------------------------
    # Structured partial-window / feature-group ablation outputs
    # --------------------------------------------------------
    write_df_to_excel_if_available(
        writer,
        "structured_window_ablation_results",
        "structured_results"
    )

    write_df_to_excel_if_available(
        writer,
        "structured_window_ablation_summary",
        "structured_summary"
    )

    write_df_to_excel_if_available(
        writer,
        "structured_window_ablation_best",
        "structured_best"
    )

    write_df_to_excel_if_available(
        writer,
        "structured_window_feature_sets",
        "structured_features"
    )

    # --------------------------------------------------------
    # Structured MAW-GME expert weights
    # --------------------------------------------------------
    write_df_to_excel_if_available(
        writer,
        "structured_maw_gme_weights",
        "maw_gme_weights"
    )

    # --------------------------------------------------------
    # Feature documentation
    # --------------------------------------------------------
    if "feature_doc_rows" in globals():
        make_excel_safe(pd.DataFrame(feature_doc_rows)).to_excel(
            writer,
            sheet_name="feature_sets",
            index=False
        )

    # --------------------------------------------------------
    # SHAP importance
    # --------------------------------------------------------
    shap_path = os.path.join(RESULT_DIR, "shap_importance_advanced.csv")
    if os.path.exists(shap_path):
        shap_df = pd.read_csv(shap_path)
        make_excel_safe(shap_df).to_excel(
            writer,
            sheet_name="shap_importance",
            index=False
        )

print("\nDONE.")
print("All results saved to Google Drive folder:", RESULT_DIR)
print("Excel report:", excel_path)
print("Important outputs:")

for fn in [
    "all_leave_one_cell_model_results.csv",
    "all_model_summary_mean_std.csv",
    "single_hi_vs_best_multi_hi_summary.csv",

    "maw_gme_leave_one_cell_results.csv",
    "maw_gme_summary_mean_std.csv",

    "structured_window_ablation_leave_one_cell_results.csv",
    "structured_window_ablation_summary_mean_std.csv",
    "structured_window_ablation_best_models.csv",
    "structured_window_ablation_feature_sets.csv",
    "structured_window_ablation_maw_gme_expert_weights.csv",

    "multi_hi_tabular_leave_one_cell_results.csv",
    "multi_hi_tabular_summary_mean_std.csv",

    "multi_hi_deep_sequence_leave_one_cell_results.csv",
    "multi_hi_deep_sequence_summary_mean_std.csv",

    "single_hi_polynomial_ekf_leave_one_cell_results.csv",
    "single_hi_polynomial_ekf_summary_mean_std.csv",

    "approx_paper_gpr_ekf_leave_one_cell_results.csv",
    "approx_paper_gpr_ekf_summary_mean_std.csv",

    "shap_importance_advanced.csv",
    "advanced_soh_experiment_report.xlsx",
]:
    p = os.path.join(RESULT_DIR, fn)
    print(" -", p, "exists=", os.path.exists(p))